<a href="https://colab.research.google.com/github/wnstj1126-debug/-/blob/main/04_split_by_bearing_py_(%2B_02_%EC%A0%84%EC%B2%B4_%EB%A3%A8%ED%94%84_%ED%86%B5%ED%95%A9).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# 04_split_by_bearing.py  (+ 02 전체 루프 통합)
# AI 김반장 LITE — 베어링 그룹 분할 + 전 파일 도메인 변환 오케스트레이션
#
# 역할:
#   1) Paderborn_raw_mat 폴더의 모든 MAT 파일명 파싱 → 메타데이터 생성
#   2) 베어링코드(bearing_code) 단위로 Train/Val/Test 분할 (★누수 절대금지)
#   3) 각 split의 파일들을 02_convert로 도메인 변환 (train만 randomize=True)
#   4) 03_build_windows가 받을 file_results 리스트 반환
#
# ★ 실행 전제: 02_convert_iis3dwb_domain.py 가 같은 세션에 import/실행되어 있어야 함
#    (convert_file_to_iis3dwb, DomainConfig 사용)
# ==============================================================================

import os
import re
import numpy as np
from dataclasses import dataclass
from typing import Optional

# ==============================================================================
# 1. 파일명 파서 — Paderborn 규칙
#    예: N09_M07_F10_K001_1.mat / N15_M01_F10_KA05_12.mat
# ==============================================================================
FNAME_RE = re.compile(
    r"^(?P<speed>N\d+)_(?P<load>M\d+)_(?P<force>F\d+)_(?P<bearing>[A-Z]+\d+)_(?P<trial>\d+)\.mat$",
    re.IGNORECASE
)

def parse_filename(fname: str) -> Optional[dict]:
    m = FNAME_RE.match(fname)
    if not m:
        return None
    d = m.groupdict()
    bearing = d["bearing"].upper()
    # 정상: K0으로 시작 (K001~K006) / 결함: KA, KI, KB...
    is_normal = bearing.startswith("K0")
    # condition_code: 속도+부하+힘 조합 (운전조건 식별용)
    condition_code = f"{d['speed']}_{d['load']}_{d['force']}".upper()
    return {
        "source_file": fname,
        "bearing_code": bearing,
        "condition_code": condition_code,
        "trial_id": d["trial"],
        "label_binary": 0 if is_normal else 1,
        "is_normal": is_normal,
    }


# ==============================================================================
# 2. 폴더 스캔 → 전체 매니페스트 생성
# ==============================================================================
def scan_dataset(mat_dir: str):
    files = sorted([f for f in os.listdir(mat_dir) if f.lower().endswith(".mat")])
    manifest, skipped = [], []
    for f in files:
        meta = parse_filename(f)
        if meta is None:
            skipped.append(f)
            continue
        meta["path"] = os.path.join(mat_dir, f)
        manifest.append(meta)

    # 요약
    bearings = sorted(set(m["bearing_code"] for m in manifest))
    normals = sorted(set(m["bearing_code"] for m in manifest if m["is_normal"]))
    faults  = sorted(set(m["bearing_code"] for m in manifest if not m["is_normal"]))
    conds   = sorted(set(m["condition_code"] for m in manifest))

    print("=" * 60)
    print("데이터셋 스캔 결과")
    print("=" * 60)
    print(f" 총 파일: {len(manifest)}개 처리 / {len(skipped)}개 스킵")
    if skipped:
        print(f"   스킵 파일(파일명 규칙 불일치): {skipped[:5]}{'...' if len(skipped)>5 else ''}")
    print(f" 베어링: {len(bearings)}종 (정상 {len(normals)} / 결함 {len(faults)})")
    print(f"   정상: {normals}")
    print(f"   결함: {faults}")
    print(f" 운전조건: {len(conds)}종 {conds}")
    return manifest, skipped


# ==============================================================================
# 3. 베어링 그룹 분할 — ★ 동일 bearing_code는 한 split에만
# ==============================================================================
@dataclass
class SplitConfig:
    seed: int = 42
    normal_test_count: int = 1     # 정상 6종 중 test 1종
    normal_val_count: int = 1      # val 1종 (나머지 4종 train)
    fault_test_ratio: float = 0.15
    fault_val_ratio: float = 0.15


def split_by_bearing(manifest: list, cfg: SplitConfig):
    rng = np.random.default_rng(cfg.seed)

    normals = sorted(set(m["bearing_code"] for m in manifest if m["is_normal"]))
    faults  = sorted(set(m["bearing_code"] for m in manifest if not m["is_normal"]))

    # --- 정상 분할 ---
    normals_sh = normals.copy(); rng.shuffle(normals_sh)
    n_te = cfg.normal_test_count
    n_va = cfg.normal_val_count
    normal_test  = normals_sh[:n_te]
    normal_val   = normals_sh[n_te:n_te + n_va]
    normal_train = normals_sh[n_te + n_va:]

    # --- 결함 분할 ---
    faults_sh = faults.copy(); rng.shuffle(faults_sh)
    f_te = max(3, int(len(faults_sh) * cfg.fault_test_ratio))
    f_va = max(3, int(len(faults_sh) * cfg.fault_val_ratio))
    fault_test  = faults_sh[:f_te]
    fault_val   = faults_sh[f_te:f_te + f_va]
    fault_train = faults_sh[f_te + f_va:]

    train_b = set(normal_train + fault_train)
    val_b   = set(normal_val + fault_val)
    test_b  = set(normal_test + fault_test)

    # --- ★ 누수 검증 (assert) ---
    assert not (train_b & val_b),  f"Train-Val 누수: {train_b & val_b}"
    assert not (train_b & test_b), f"Train-Test 누수: {train_b & test_b}"
    assert not (val_b & test_b),   f"Val-Test 누수: {val_b & test_b}"
    # 각 split에 정상/결함 모두 존재하는지
    for name, s in [("Train", train_b), ("Val", val_b), ("Test", test_b)]:
        has_n = any(b.startswith("K0") for b in s)
        has_f = any(not b.startswith("K0") for b in s)
        assert has_n and has_f, f"{name}에 정상 또는 결함 베어링이 없음: {sorted(s)}"

    print("\n" + "=" * 60)
    print(f"베어링 분할 (seed={cfg.seed}) — 누수 검증 통과 ✅")
    print("=" * 60)
    print(f" Train({len(train_b)}종): {sorted(train_b)}")
    print(f" Val  ({len(val_b)}종): {sorted(val_b)}")
    print(f" Test ({len(test_b)}종): {sorted(test_b)}")

    # 파일을 split별로 배정
    def files_of(bearing_set):
        return [m for m in manifest if m["bearing_code"] in bearing_set]

    return {
        "train_bearings": train_b, "val_bearings": val_b, "test_bearings": test_b,
        "train_files": files_of(train_b),
        "val_files":   files_of(val_b),
        "test_files":  files_of(test_b),
    }


# ==============================================================================
# 4. ★ 02 전체 루프 — 각 split 파일을 도메인 변환
#    train: randomize=True (증강) / val·test: randomize=False (재현성)
# ==============================================================================
def convert_split(files: list, is_train: bool, base_seed: int,
                  declared_unit: str = "g", channel: str = "vibration_1"):
    """
    files: manifest 항목 리스트 (path, bearing_code 등 포함)
    반환: 03_build_windows.build_windows_from_files 입력 형식 리스트
    """
    from_02_ok = True
    try:
        # 02단계 함수 (같은 세션에 정의돼 있어야 함)
        _ = convert_file_to_iis3dwb  # noqa
        _ = DomainConfig             # noqa
    except NameError:
        from_02_ok = False
    if not from_02_ok:
        raise RuntimeError("★ 02_convert_iis3dwb_domain.py를 먼저 실행하세요 "
                           "(convert_file_to_iis3dwb, DomainConfig 필요)")

    results = []
    tag = "TRAIN(증강)" if is_train else "EVAL(고정)"
    print(f"\n[{tag}] 도메인 변환 시작 — {len(files)}개 파일")

    for i, m in enumerate(files):
        # ★ 파일마다 다른 seed → 증강 다양성 확보(재현 가능)
        file_seed = base_seed + i
        cfg = DomainConfig(randomize=is_train, seed=file_seed,
                           lpf_causal=True, apply_offset_gravity=True)

        res = convert_file_to_iis3dwb(m["path"], cfg,
                                      declared_unit=declared_unit, channel=channel)
        if res is None:
            print(f"   [SKIP] {m['source_file']}")
            continue

        results.append({
            "counts_int16": res.counts_int16,
            "label": m["label_binary"],
            "bearing_code": m["bearing_code"],
            "condition_code": m["condition_code"],
            "trial_id": m["trial_id"],
            "source_file": m["source_file"],
            "effect_meta": res.effect_meta,
            "clip_ratio": res.quant_report["clip_ratio"],
        })

        if (i + 1) % 20 == 0:
            print(f"   진행 {i+1}/{len(files)}...")

    # 클리핑 과다 경고
    high_clip = [r["source_file"] for r in results if r["clip_ratio"] > 0.001]
    if high_clip:
        print(f"   ⚠ 클리핑 0.1% 초과 {len(high_clip)}개: {high_clip[:3]}...")
    print(f"   완료: {len(results)}개 변환")
    return results


# ==============================================================================
# 5. 전체 오케스트레이션 (04 → 02루프 → 03 연결 직전까지)
# ==============================================================================
def build_split_and_convert(mat_dir: str, seed: int = 42,
                            declared_unit: str = "g"):
    # (1) 스캔
    manifest, _ = scan_dataset(mat_dir)
    # (2) 분할
    split = split_by_bearing(manifest, SplitConfig(seed=seed))
    # (3) 02 전체 루프 변환
    train_results = convert_split(split["train_files"], is_train=True,
                                  base_seed=seed, declared_unit=declared_unit)
    val_results   = convert_split(split["val_files"],   is_train=False,
                                  base_seed=seed + 10000, declared_unit=declared_unit)
    test_results  = convert_split(split["test_files"],  is_train=False,
                                  base_seed=seed + 20000, declared_unit=declared_unit)

    print("\n" + "=" * 60)
    print("변환 완료 요약")
    print("=" * 60)
    for name, r in [("Train", train_results), ("Val", val_results), ("Test", test_results)]:
        n0 = sum(1 for x in r if x["label"] == 0)
        n1 = sum(1 for x in r if x["label"] == 1)
        print(f" {name}: 파일 {len(r)}개 (정상 {n0} / 결함 {n1})")

    return {
        "manifest": manifest,
        "split": split,
        "train_results": train_results,
        "val_results": val_results,
        "test_results": test_results,
    }


# ==============================================================================
# 6. 사용 예시 (03과 연결)
# ==============================================================================
if __name__ == "__main__":
    MAT_DIR = "/content/drive/My Drive/Colab Notebooks/Paderborn_raw_mat"

    # 04 + 02루프 실행
    bundle = build_split_and_convert(MAT_DIR, seed=42, declared_unit="g")

    # ── 여기서부터 03_build_windows.py로 연결 ──
    # from importlib import ...  (또는 03 코드를 같은 셀에 정의)
    # train_ds, val_ds, test_ds = make_dataset(
    #     bundle["train_results"],
    #     bundle["val_results"],
    #     bundle["test_results"],
    #     window=2048,
    #     scaling_method="symmetric"
    # )
    print("\n✅ 04+02 완료. 다음: 03_build_windows.make_dataset() 호출")
